In [1]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_style("whitegrid")


In [ ]:

df = pd.read_csv("Mall_Customers.csv")

df.head(10)


In [ ]:

print("Dataset Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)


In [ ]:

print("Missing Values:")
print(df.isnull().sum())

print("\nDuplicate Records:", df.duplicated().sum())


In [ ]:

income = df["Annual Income (k$)"]

print("Average Annual Income:", income.mean())
print("Median Annual Income:", income.median())
print("Minimum Annual Income:", income.min())
print("Maximum Annual Income:", income.max())


In [ ]:

plt.figure(figsize=(8, 5))

plt.hist(df["Age"], bins=15, edgecolor="black")
plt.title("Age Distribution of Customers")
plt.xlabel("Age")
plt.ylabel("Number of Customers")

plt.show()


In [ ]:

plt.figure(figsize=(8, 4))

sns.boxplot(x=df["Annual Income (k$)"])
plt.title("Annual Income - Box Plot")
plt.xlabel("Annual Income (k$)")

plt.show()


In [ ]:

plt.figure(figsize=(8, 5))

plt.scatter(
    df["Annual Income (k$)"],
    df["Spending Score (1-100)"],
    alpha=0.7
)

plt.title("Annual Income vs Spending Score")
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")

plt.show()


In [ ]:

X = df[["Annual Income (k$)", "Spending Score (1-100)"]]

X.head()


In [ ]:

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled[:5]


In [ ]:

inertia_values = []

for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia_values.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))

plt.plot(range(1, 11), inertia_values, marker="o")
plt.title("Elbow Method")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia (WCSS)")
plt.xticks(range(1, 11))

plt.show()


In [ ]:

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)

df["Cluster"] = kmeans.fit_predict(X_scaled)

df.head(10)


In [ ]:

print("Customers in Each Cluster:")
print(df["Cluster"].value_counts().sort_index())


In [ ]:

centroids = scaler.inverse_transform(kmeans.cluster_centers_)

centroids_df = pd.DataFrame(
    centroids,
    columns=["Annual Income (k$)", "Spending Score (1-100)"]
)

centroids_df.index.name = "Cluster"

centroids_df


In [ ]:

plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=df,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    hue="Cluster",
    palette="viridis",
    s=80
)

plt.scatter(
    centroids[:, 0],
    centroids[:, 1],
    color="red",
    marker="X",
    s=250,
    label="Centroids"
)

plt.title("Customer Segmentation Using K-Means")
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.legend()

plt.show()


In [ ]:

score = silhouette_score(X_scaled, df["Cluster"])

print("Silhouette Score:", score)


In [ ]:

comparison = []

for k in [4, 5]:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)

    comparison.append({
        "K": k,
        "Inertia": model.inertia_,
        "Silhouette Score": silhouette_score(X_scaled, labels)
    })

comparison_df = pd.DataFrame(comparison)

comparison_df


In [ ]:

cluster_summary = df.groupby("Cluster").agg(
    Customer_Count=("CustomerID", "count"),
    Average_Age=("Age", "mean"),
    Average_Income=("Annual Income (k$)", "mean"),
    Average_Spending_Score=("Spending Score (1-100)", "mean")
).round(2)

cluster_summary


In [ ]:

high_income_threshold = df["Annual Income (k$)"].median()
high_spending_threshold = df["Spending Score (1-100)"].median()

high_value_segments = cluster_summary[
    (cluster_summary["Average_Income"] > high_income_threshold) &
    (cluster_summary["Average_Spending_Score"] > high_spending_threshold)
]

print("High-Income and High-Spending Segments:")
high_value_segments
